In [ ]:
import pandas as pd
import plotly.express as px

file_path = '/content/Acceptance - Trust Scale (Responses) - Form responses 1.csv'
df = pd.read_csv(file_path)
df.drop(columns=['Timestamp'], inplace=True)
df.head()

,Participant ID,Choose the Experiment Condition,Personality Trait,I find such a system...,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,Unnamed: 10,Unnamed: 11,Unnamed: 12,I trust that the autonomous vehicle will operate safely and effectively under various conditions [Please click one that applies]
0,1,A,High Openness - Low Neuroticism,1,1,5,1,1,5,1,5,2,Agree
1,1,C,High Openness - Low Neuroticism,1,1,5,1,1,5,1,5,3,Agree
2,2,A,High Openness - Low Neuroticism,1,1,5,1,1,5,1,5,1,Agree
3,2,C,High Openness - Low Neuroticism,1,1,5,1,1,5,1,5,1,Agree
4,3,A,High Neuroticism - Low Openness,1,2,4,2,2,5,2,5,2,Agree


In [ ]:
print(df.columns)

Index(['Participant ID', 'Choose the Experiment Condition',
       'Personality Trait', 'I find such a system...', 'Unnamed: 5',
       'Unnamed: 6', 'Unnamed: 7', 'Unnamed: 8', 'Unnamed: 9', 'Unnamed: 10',
       'Unnamed: 11', 'Unnamed: 12',
       'I trust that the autonomous vehicle will operate safely and effectively under various conditions [Please click one that applies]'],
      dtype='object')


In [ ]:
df.rename(columns={
    'Participant ID': 'participant_id',
    'Choose the Experiment Condition': 'experiment_condition',
    'Personality Trait': 'personality_trait',
    'I find such a system...': 'useful/useless',
    'Unnamed: 5': 'pleasent/unpleasent',
    'Unnamed: 6': 'bad/good',
    'Unnamed: 7': 'nice/annoying',
    'Unnamed: 8': 'effective/superflous',
    'Unnamed: 9': 'irritating/likeable',
    'Unnamed: 10': 'assisting/worthless',
    'Unnamed: 11': 'undesirable/desirable',
    'Unnamed: 12': 'raising alertness/sleep-inducing',
    'I trust that the autonomous vehicle will operate safely and effectively under various conditions [Please click one that applies]': 'trust-safety-effectiveness'
}, inplace=True)

print(df.columns)

Index(['participant_id', 'experiment_condition', 'personality_trait',
       'useful/useless', 'pleasent/unpleasent', 'bad/good', 'nice/annoying',
       'effective/superflous', 'irritating/likeable', 'assisting/worthless',
       'undesirable/desirable', 'raising alertness/sleep-inducing',
       'trust-safety-effectiveness'],
      dtype='object')


In [ ]:
df.head(10)

,participant_id,experiment_condition,personality_trait,useful/useless,pleasent/unpleasent,bad/good,nice/annoying,effective/superflous,irritating/likeable,assisting/worthless,undesirable/desirable,raising alertness/sleep-inducing,trust-safety-effectiveness
0,1,A,High Openness - Low Neuroticism,1,1,5,1,1,5,1,5,2,Agree
1,1,C,High Openness - Low Neuroticism,1,1,5,1,1,5,1,5,3,Agree
2,2,A,High Openness - Low Neuroticism,1,1,5,1,1,5,1,5,1,Agree
3,2,C,High Openness - Low Neuroticism,1,1,5,1,1,5,1,5,1,Agree
4,3,A,High Neuroticism - Low Openness,1,2,4,2,2,5,2,5,2,Agree
5,3,B,High Neuroticism - Low Openness,1,2,5,2,2,4,2,4,2,Agree
6,4,A,High Neuroticism - Low Openness,3,3,3,2,3,4,1,4,1,Agree
7,4,C,High Neuroticism - Low Openness,3,2,3,3,3,4,2,3,3,Agree
8,5,A,High Neuroticism - Low Openness,3,3,3,3,2,2,3,3,3,Neutral
9,5,B,High Neuroticism - Low Openness,3,4,3,3,4,2,4,3,3,Neutral


In [ ]:
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Mapping responses to numeric values
response_map = {
    'Strongly Disagree': -2,
    'Disagree': -1,
    'Neutral': 0,
    'Agree': 1,
    'Strongly Agree': 2
}

df['Response_Numeric'] = df['trust-safety-effectiveness'].map(response_map)

# Calculate percentages for each response by condition
df_percent = df.groupby('experiment_condition')['trust-safety-effectiveness'].value_counts(normalize=True).unstack().fillna(0) * 100

# Order the columns
df_percent = df_percent[['Strongly Disagree', 'Disagree', 'Neutral', 'Agree', 'Strongly Agree']]

# Create subplots
fig = make_subplots(rows=2, cols=1, subplot_titles=("Stacked Bar Chart", "Diverging Stacked Bar Chart"))

# Stacked Bar Chart
for response in df_percent.columns:
    fig.add_trace(
        go.Bar(name=response, x=df_percent.index, y=df_percent[response], text=df_percent[response].round(1), textposition='inside'),
        row=1, col=1
    )

# Prepare data for Diverging Stacked Bar Chart
neutral = df_percent['Neutral']
left = df_percent[['Strongly Disagree', 'Disagree']].sum(axis=1)
right = df_percent[['Agree', 'Strongly Agree']].sum(axis=1)

fig.add_trace(go.Bar(name='Disagree', x=df_percent.index, y=-left, marker_color='#e74c3c', text=left.round(1), textposition='inside'), row=2, col=1)
fig.add_trace(go.Bar(name='Neutral', x=df_percent.index, y=-neutral/2, marker_color='#7f8c8d', showlegend=False), row=2, col=1)
fig.add_trace(go.Bar(name='Neutral', x=df_percent.index, y=neutral/2, marker_color='#7f8c8d'), row=2, col=1)
fig.add_trace(go.Bar(name='Agree', x=df_percent.index, y=right, marker_color='#2ecc71', text=right.round(1), textposition='inside'), row=2, col=1)

# Update layout
fig.update_layout(
    barmode='stack',
    title_text="Trust in Autonomous Vehicle Safety and Effectiveness",
    height=2000,
    width=1400,  # Increased width for better visualization
    legend_title_text='Response',
    xaxis_title="Experiment Condition",
    yaxis_title="Percentage",
    yaxis2_title="Percentage",
    yaxis2_range=[-100, 100],
    plot_bgcolor='white',
    paper_bgcolor='white'
)

# Show plot
fig.show()


In [ ]:
df

,participant_id,experiment_condition,personality_trait,useful/useless,pleasent/unpleasent,bad/good,nice/annoying,effective/superflous,irritating/likeable,assisting/worthless,undesirable/desirable,raising alertness/sleep-inducing,trust-safety-effectiveness,Response_Numeric
0,1,A,High Openness - Low Neuroticism,1,1,5,1,1,5,1,5,2,Agree,1
1,1,C,High Openness - Low Neuroticism,1,1,5,1,1,5,1,5,3,Agree,1
2,2,A,High Openness - Low Neuroticism,1,1,5,1,1,5,1,5,1,Agree,1
3,2,C,High Openness - Low Neuroticism,1,1,5,1,1,5,1,5,1,Agree,1
4,3,A,High Neuroticism - Low Openness,1,2,4,2,2,5,2,5,2,Agree,1
5,3,B,High Neuroticism - Low Openness,1,2,5,2,2,4,2,4,2,Agree,1
6,4,A,High Neuroticism - Low Openness,3,3,3,2,3,4,1,4,1,Agree,1
7,4,C,High Neuroticism - Low Openness,3,2,3,3,3,4,2,3,3,Agree,1
8,5,A,High Neuroticism - Low Openness,3,3,3,3,2,2,3,3,3,Neutral,0
9,5,B,High Neuroticism - Low Openness,3,4,3,3,4,2,4,3,3,Neutral,0


In [ ]:
df_trust = df["experiment_condition", "response"]

In [ ]:
condition_A_df = df[df['experiment_condition'] == 'A']
condition_B_df = df[df['experiment_condition'] == 'B']
condition_C_df = df[df['experiment_condition'] == 'C']

In [ ]:
import plotly.express as px

# List of conditions
conditions = {
    'A': condition_A_df,
    'B': condition_B_df,
    'C': condition_C_df
}

# List of column names
columns = [
    'useful/useless',
    'pleasent/unpleasent',
    'bad/good',
    'nice/annoying',
    'effective/superflous',
    'irritating/likeable',
    'assisting/worthless',
    'undesirable/desirable',
    'raising alertness/sleep-inducing',
    'trust-safety-effectiveness'
]

# Colors for each condition
colors = {
    'A': '#009596',
    'B': '#F4C145',
    'C': '#A30000'
}

# Loop through each condition and each column
for condition, df in conditions.items():
    for column in columns:
        # Count the number of responses for each rating in the column
        rating_counts = df[column].value_counts().sort_index()

        # Create a bar chart for the condition and column
        fig = px.bar(
            rating_counts,
            x=rating_counts.index,
            y=rating_counts.values,
            labels={'x': 'Rating', 'y': 'Count'},
            title=f'{column.replace("_", " ").title()} Ratings for Condition {condition}',
            color_discrete_sequence=[colors[condition]]
        )

        fig.update_layout(
            plot_bgcolor='white',
            paper_bgcolor='white'
        )

        # Show the figure
        fig.show()

In [ ]:
condition_A_df

,participant_id,experiment_condition,personality_trait,useful/useless,pleasent/unpleasent,bad/good,nice/annoying,effective/superflous,irritating/likeable,assisting/worthless,undesirable/desirable,raising alertness/sleep-inducing,trust-safety-effectiveness,Response_Numeric
0,1,A,High Openness - Low Neuroticism,1,1,5,1,1,5,1,5,2,Agree,1
2,2,A,High Openness - Low Neuroticism,1,1,5,1,1,5,1,5,1,Agree,1
4,3,A,High Neuroticism - Low Openness,1,2,4,2,2,5,2,5,2,Agree,1
6,4,A,High Neuroticism - Low Openness,3,3,3,2,3,4,1,4,1,Agree,1
8,5,A,High Neuroticism - Low Openness,3,3,3,3,2,2,3,3,3,Neutral,0
10,6,A,High Neuroticism - Low Openness,2,3,4,3,4,4,2,2,3,Strongly Disagree,-2
12,7,A,High Openness - Low Neuroticism,2,2,5,2,1,4,1,5,2,Agree,1
14,8,A,High Openness - Low Neuroticism,1,2,5,1,1,5,1,4,1,Strongly Agree,2
16,9,A,High Neuroticism - Low Openness,1,1,5,1,5,5,1,5,4,Agree,1
18,10,A,High Openness - Low Neuroticism,1,1,4,2,1,4,2,5,3,Strongly Agree,2


In [ ]:
condition_B_df

,participant_id,experiment_condition,personality_trait,useful/useless,pleasent/unpleasent,bad/good,nice/annoying,effective/superflous,irritating/likeable,assisting/worthless,undesirable/desirable,raising alertness/sleep-inducing,trust-safety-effectiveness,Response_Numeric
5,3,B,High Neuroticism - Low Openness,1,2,5,2,2,4,2,4,2,Agree,1
9,5,B,High Neuroticism - Low Openness,3,4,3,3,4,2,4,3,3,Neutral,0
13,7,B,High Openness - Low Neuroticism,2,2,5,1,2,5,2,4,4,Agree,1
17,9,B,High Neuroticism - Low Openness,2,3,3,3,4,4,2,4,4,Agree,1
21,11,B,High Openness - Low Neuroticism,5,4,2,4,4,2,4,2,4,Strongly Disagree,-2
27,15,B,High Neuroticism - Low Openness,4,5,2,4,3,1,5,1,1,Disagree,-1
31,17,B,High Neuroticism - Low Openness,5,4,1,5,3,2,4,4,5,Strongly Disagree,-2
34,12,B,High Openness - Low Neuroticism,4,4,3,4,4,3,4,2,4,Agree,1
36,19,B,High Neuroticism - Low Openness,3,3,4,2,2,4,4,3,4,Neutral,0
37,13,B,High Neuroticism - Low Openness,5,4,2,4,4,2,4,2,2,Disagree,-1


In [ ]:
condition_C_df

,participant_id,experiment_condition,personality_trait,useful/useless,pleasent/unpleasent,bad/good,nice/annoying,effective/superflous,irritating/likeable,assisting/worthless,undesirable/desirable,raising alertness/sleep-inducing,trust-safety-effectiveness,Response_Numeric
1,1,C,High Openness - Low Neuroticism,1,1,5,1,1,5,1,5,3,Agree,1
3,2,C,High Openness - Low Neuroticism,1,1,5,1,1,5,1,5,1,Agree,1
7,4,C,High Neuroticism - Low Openness,3,2,3,3,3,4,2,3,3,Agree,1
11,6,C,High Neuroticism - Low Openness,2,4,2,4,4,2,4,3,3,Agree,1
15,8,C,High Openness - Low Neuroticism,2,2,5,2,2,5,2,4,2,Agree,1
19,10,C,High Openness - Low Neuroticism,1,2,4,1,1,4,1,5,3,Strongly Agree,2
23,12,C,High Openness - Low Neuroticism,2,4,2,4,4,2,2,4,2,Disagree,-1
25,14,C,High Neuroticism - Low Openness,1,2,5,2,2,4,2,5,2,Strongly Agree,2
29,16,C,High Openness - Low Neuroticism,2,1,4,1,2,5,2,4,1,Agree,1
33,18,C,High Openness - Low Neuroticism,1,2,5,1,1,5,2,4,3,Agree,1


In [ ]:
print(condition_A_df.columns)

Index(['participant_id', 'experiment_condition', 'personality_trait',
       'useful/useless', 'pleasent/unpleasent', 'bad/good', 'nice/annoying',
       'effective/superflous', 'irritating/likeable', 'assisting/worthless',
       'undesirable/desirable', 'raising alertness/sleep-inducing',
       'trust-safety-effectiveness', 'Response_Numeric'],
      dtype='object')


In [ ]:
import pandas as pd

# Define the response to score mapping
response_to_score = {1: 2, 2: 1, 3: 0, 4: -1, 5: -2}

# Columns to be mirrored
mirrored_columns = ['bad/good', 'irritating/likeable', 'undesirable/desirable']

# Function to apply the scoring to the columns
def apply_scoring(df, columns, mirrored_columns):
    # Apply the regular scoring
    for col in columns:
        if col in mirrored_columns:
            df[col] = df[col].map(response_to_score).apply(lambda x: -x)
        else:
            df[col] = df[col].map(response_to_score)
    return df

# List of columns to be scored
columns_to_score = [
    'useful/useless', 'pleasent/unpleasent', 'bad/good', 'nice/annoying',
    'effective/superflous', 'irritating/likeable', 'assisting/worthless',
    'undesirable/desirable', 'raising alertness/sleep-inducing'
]

# Apply scoring to condition A, B, and C dataframes
condition_A_df = apply_scoring(condition_A_df, columns_to_score, mirrored_columns)
condition_B_df = apply_scoring(condition_B_df, columns_to_score, mirrored_columns)
condition_C_df = apply_scoring(condition_C_df, columns_to_score, mirrored_columns)

# Function to calculate the usefulness and satisfying scores
def calculate_scores(df):
    df['usefulness'] = df['useful/useless'] + df['bad/good'] + df['effective/superflous'] + df['assisting/worthless'] + df['raising alertness/sleep-inducing']
    df['usefulness'] /= 5.0
    df['satisfying'] = df['pleasent/unpleasent'] + df['nice/annoying'] + df['irritating/likeable'] + df['undesirable/desirable']
    df['satisfying'] /= 4.0
    return df[['usefulness', 'satisfying']]

# Calculate scores for each condition
scores_A = calculate_scores(condition_A_df)
scores_B = calculate_scores(condition_B_df)
scores_C = calculate_scores(condition_C_df)

# Combine the scores into a single dataframe for visualization
combined_scores = pd.concat([
    scores_A.assign(condition='Functional UI'),
    scores_B.assign(condition='No UI'),
    scores_C.assign(condition='Placebo UI')
])

# Display the combined scores dataframe
print(combined_scores)

# Visualization using Plotly
import plotly.express as px

# Plot for usefulness
fig_usefulness = px.box(
    combined_scores,
    x='condition',
    y='usefulness',
    title='Usefulness Ratings by Condition',
    labels={'usefulness': 'Usefulness', 'condition': 'Condition'},
    color='condition',
    color_discrete_sequence=['#8BC1F7', '#F9E0A2', '#C9190B']
)
fig_usefulness.show()

# Plot for satisfying
fig_satisfying = px.box(
    combined_scores,
    x='condition',
    y='satisfying',
    title='Satisfying Ratings by Condition',
    labels={'satisfying': 'Satisfying', 'condition': 'Condition'},
    color='condition',
    color_discrete_sequence=['#8BC1F7', '#F9E0A2', '#C9190B']
)
fig_satisfying.show()


<ipython-input-13-b6c8997cff5e>:16: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

<ipython-input-13-b6c8997cff5e>:14: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

<ipython-input-13-b6c8997cff5e>:33: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

<ipython-input-13-b6c8

    usefulness  satisfying      condition
0          1.8        2.00  Functional UI
2          2.0        2.00  Functional UI
4          1.2        1.50  Functional UI
6          0.8        0.75  Functional UI
8          0.2       -0.25  Functional UI
10         0.4        0.00  Functional UI
12         1.6        1.25  Functional UI
14         2.0        1.50  Functional UI
16         0.6        2.00  Functional UI
18         1.2        1.50  Functional UI
20         0.4        0.75  Functional UI
22         0.0       -0.50  Functional UI
24         1.4        0.75  Functional UI
26         1.6        1.50  Functional UI
28         1.2        1.25  Functional UI
30         1.4        1.25  Functional UI
32         0.8        1.25  Functional UI
35         0.8        0.75  Functional UI
38         0.8        1.00  Functional UI
40         1.4        1.50  Functional UI
42         1.2        1.75  Functional UI
44         2.0        2.00  Functional UI
46         1.6        2.00  Functi

In [ ]:
import plotly.express as px
import pandas as pd

# Combine the 'trust-safety-effectiveness' ratings from all conditions
combined_trust_safety_df = pd.concat([
    condition_A_df[['trust-safety-effectiveness']].assign(condition='Functional UI'),
    condition_B_df[['trust-safety-effectiveness']].assign(condition='No UI'),
    condition_C_df[['trust-safety-effectiveness']].assign(condition='Placebo UI')
])

# Convert the 'trust-safety-effectiveness' column to categorical with correct order
combined_trust_safety_df['trust-safety-effectiveness'] = pd.Categorical(
    combined_trust_safety_df['trust-safety-effectiveness'],
    categories=['Strongly Disagree', 'Disagree', 'Neutral', 'Agree', 'Strongly Agree'],
    ordered=True
)

# Create a box plot to compare the 'trust-safety-effectiveness' ratings across conditions
fig_trust_safety = px.box(
    combined_trust_safety_df,
    x='condition',
    y='trust-safety-effectiveness',
    title='Trust-Safety-Effectiveness Ratings by Condition',
    labels={'trust-safety-effectiveness': 'Trust-Safety-Effectiveness', 'condition': 'Condition'},
    color='condition',
    color_discrete_sequence=['#8BC1F7', '#F9E0A2', '#C9190B']
)

# Customize the layout for better visibility
fig_trust_safety.update_layout(
    plot_bgcolor='white',
    paper_bgcolor='white',
    yaxis=dict(categoryorder='array', categoryarray=['Strongly Disagree', 'Disagree', 'Neutral', 'Agree', 'Strongly Agree']),
    xaxis=dict(
        tickvals=['Functional UI', 'No UI', 'Placebo UI'],
        ticktext=['Functional UI', 'No UI', 'Placebo UI']
    )
)

# Show the figure
fig_trust_safety.show()


In [ ]:
import pandas as pd
from scipy.stats import f_oneway
import plotly.express as px

# Define the Acceptance Scale data
data = {
    'ParticipantID': [0, 2, 4, 6, 8, 10, 12, 14, 16, 18, 20, 22, 24, 26, 28, 30, 32, 35, 38, 40, 42, 44, 46, 48, 50, 52,
                      5, 9, 13, 17, 21, 27, 31, 34, 36, 37, 41, 45, 49, 53,
                      1, 3, 7, 11, 15, 19, 23, 25, 29, 33, 39, 43, 47, 51],
    'usefulness': [1.8, 2.0, 1.2, 0.8, 0.2, 0.4, 1.6, 2.0, 0.6, 1.2, 0.4, 0.0, 1.4, 1.6, 1.2, 1.4, 0.8, 0.8, 0.8, 1.4, 1.2, 2.0, 1.6, 0.2, 1.4, 1.0,
                   1.4, -0.4, 0.8, 0.0, -1.2, -0.4, -1.4, -0.8, 0.0, -0.8, 1.4, 1.8, -0.8, -0.8,
                   1.6, 2.0, 0.2, -0.4, 1.2, 1.4, 0.2, 1.4, 1.2, 1.4, 0.2, -0.4, 1.6, -1.4],
    'satisfying': [2.0, 2.0, 1.5, 0.75, -0.25, 0.0, 1.25, 1.5, 2.0, 1.5, 0.75, -0.5, 0.75, 1.5, 1.25, 1.25, 1.25, 0.75, 1.0, 1.5, 1.75, 2.0, 2.0, 1.0, 2.0, 0.5,
                   1.0, -0.5, 1.5, 0.5, -1.0, -1.75, -0.75, -0.75, 0.5, -1.0, 2.0, 2.0, -0.5, -0.75,
                   2.0, 2.0, 0.5, -0.75, 1.25, 1.5, -0.5, 1.25, 1.75, 1.5, 0.5, 0.0, 2.0, -0.25],
    'condition': ['Functional UI']*26 + ['No UI']*14 + ['Placebo UI']*14
}

# Create DataFrame
combined_acceptance_df = pd.DataFrame(data)

# Perform ANOVA for each score category
score_categories = ['usefulness', 'satisfying']

anova_results = {}
for category in score_categories:
    condition_a_values = combined_acceptance_df[combined_acceptance_df['condition'] == 'Functional UI'][category].dropna()
    condition_b_values = combined_acceptance_df[combined_acceptance_df['condition'] == 'No UI'][category].dropna()
    condition_c_values = combined_acceptance_df[combined_acceptance_df['condition'] == 'Placebo UI'][category].dropna()

    f_stat, p_value = f_oneway(condition_a_values, condition_b_values, condition_c_values)
    anova_results[category] = {'f_stat': f_stat, 'p_value': p_value}

# Display the results
for category, results in anova_results.items():
    print(f"ANOVA results for {category}: F-statistic = {results['f_stat']}, P-value = {results['p_value']}")

# Interpret the results
alpha = 0.05
for category, results in anova_results.items():
    if results['p_value'] < alpha:
        print(f"{category}: Significant differences found (p-value = {results['p_value']})")
    else:
        print(f"{category}: No significant differences found (p-value = {results['p_value']})")

# Visualization using Plotly
# Define colors for each condition
colors = {
    'Functional UI': '#8BC1F7',
    'No UI': '#F9E0A2',
    'Placebo UI': '#C9190B'
}

# Create box plots for each score category
for category in score_categories:
    fig = px.box(
        combined_acceptance_df,
        x='condition',
        y=category,
        title=f'{category.capitalize()} Ratings by Condition',
        labels={category: category.capitalize(), 'condition': 'Condition'},
        color='condition',
        color_discrete_map=colors
    )

    # Customize the layout for better visibility
    fig.update_layout(
        plot_bgcolor='white',
        paper_bgcolor='white',
        boxmode='group',
        xaxis_title='Condition',
        yaxis_title=category.capitalize(),
        font=dict(size=12),
        title_x=0.5,
        xaxis=dict(
            tickmode='array',
            tickvals=[-0.25, 1, 2.25],
            ticktext=['Functional UI (A)', 'No UI (B)', 'Placebo UI (C)']),
        hoverlabel=dict(font_color='white')
      )

    fig.show()

ANOVA results for usefulness: F-statistic = 9.48659370989867, P-value = 0.00031425502293644576
ANOVA results for satisfying: F-statistic = 7.171153108136687, P-value = 0.0018012782169897635
usefulness: Significant differences found (p-value = 0.00031425502293644576)
satisfying: Significant differences found (p-value = 0.0018012782169897635)


In [ ]:
combined_scores

,usefulness,satisfying,condition
0,1.8,2.00,Functional UI
2,2.0,2.00,Functional UI
4,1.2,1.50,Functional UI
6,0.8,0.75,Functional UI
8,0.2,-0.25,Functional UI
10,0.4,0.00,Functional UI
12,1.6,1.25,Functional UI
14,2.0,1.50,Functional UI
16,0.6,2.00,Functional UI
18,1.2,1.50,Functional UI


In [ ]:
import pandas as pd
from scipy.stats import f_oneway
from statsmodels.stats.multicomp import pairwise_tukeyhsd
import plotly.express as px

combined_scores['Sum of Results'] = combined_scores[['usefulness', 'satisfying']].sum(axis=1)

print(combined_scores)

# Perform ANOVA for the combined score
f_stat, p_value = f_oneway(combined_scores[combined_scores['condition'] == 'Functional UI']['Sum of Results'],
                           combined_scores[combined_scores['condition'] == 'No UI']['Sum of Results'],
                           combined_scores[combined_scores['condition'] == 'Placebo UI']['Sum of Results'])
print(f"ANOVA results for Sum of Results: F-statistic = {f_stat}, P-value = {p_value}")

    usefulness  satisfying      condition  Sum of Results
0          1.8        2.00  Functional UI            3.80
2          2.0        2.00  Functional UI            4.00
4          1.2        1.50  Functional UI            2.70
6          0.8        0.75  Functional UI            1.55
8          0.2       -0.25  Functional UI           -0.05
10         0.4        0.00  Functional UI            0.40
12         1.6        1.25  Functional UI            2.85
14         2.0        1.50  Functional UI            3.50
16         0.6        2.00  Functional UI            2.60
18         1.2        1.50  Functional UI            2.70
20         0.4        0.75  Functional UI            1.15
22         0.0       -0.50  Functional UI           -0.50
24         1.4        0.75  Functional UI            2.15
26         1.6        1.50  Functional UI            3.10
28         1.2        1.25  Functional UI            2.45
30         1.4        1.25  Functional UI            2.65
32         0.8

In [ ]:
# Visualization with box plots
fig = px.box(
    combined_scores,
    x='condition',
    y='Sum of Results',
    title='Sum of Results Ratings by Condition',
    labels={'Sum of Results': 'Sum of Results', 'condition': 'Condition'},
    color='condition',
    color_discrete_map={
        'Functional UI': '#8BC1F7',
        'No UI': '#F9E0A2',
        'Placebo UI': '#C9190B'
    }
)

fig.update_layout(
    plot_bgcolor='white',
    paper_bgcolor='white',
    boxmode='group',
    xaxis_title='Condition',
    yaxis_title='Sum of Results',
    font=dict(size=18),
    title_x=0.5,
    xaxis=dict(
        tickmode='array',
        tickvals=[-0.25, 1, 2.25],
        ticktext=['Functional UI', 'No UI', 'Placebo UI'])
)

fig.show()

In [ ]:
# Perform Tukey's HSD test
tukey_result = pairwise_tukeyhsd(endog=combined_scores['Sum of Results'],
                                 groups=combined_scores['condition'],
                                 alpha=0.05)
print(tukey_result)

      Multiple Comparison of Means - Tukey HSD, FWER=0.05      
    group1      group2   meandiff p-adj   lower   upper  reject
---------------------------------------------------------------
Functional UI      No UI  -2.3577 0.0003 -3.7141 -1.0013   True
Functional UI Placebo UI  -0.6684 0.4647 -2.0248   0.688  False
        No UI Placebo UI   1.6893 0.0293  0.1428  3.2358   True
---------------------------------------------------------------


In [ ]:
# Perform Tukey's HSD test
tukey_result = pairwise_tukeyhsd(endog=combined_scores['Sum of Results'],
                                 groups=combined_scores['condition'],
                                 alpha=0.05)
print(tukey_result)

      Multiple Comparison of Means - Tukey HSD, FWER=0.05      
    group1      group2   meandiff p-adj   lower   upper  reject
---------------------------------------------------------------
Functional UI      No UI  -2.3577 0.0003 -3.7141 -1.0013   True
Functional UI Placebo UI  -0.6684 0.4647 -2.0248   0.688  False
        No UI Placebo UI   1.6893 0.0293  0.1428  3.2358   True
---------------------------------------------------------------


In [ ]:
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt
import seaborn as sns

# Function to perform Levene's Test for homogeneity of variances
def levene_test(data, group_column, value_column):
    groups = data[group_column].unique()
    group_data = [data[data[group_column] == group][value_column] for group in groups]
    stat, p = stats.levene(*group_data)
    print(f"Levene's Test for {value_column} - Statistic: {stat}, p-value: {p}")

# Check homogeneity of variances for Acceptance Scale variables
acceptance_columns = ['usefulness', 'satisfying']
for column in acceptance_columns:
    levene_test(combined_scores, 'condition', column)


Levene's Test for usefulness - Statistic: 2.161739461581276, p-value: 0.12555895578285825
Levene's Test for satisfying - Statistic: 2.2858606185346355, p-value: 0.1120124647246718


In [ ]:
import pandas as pd
from scipy.stats import f_oneway
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# (Keep the data and ANOVA calculations as they are)

# Create subplots
fig = make_subplots(rows=1, cols=2, subplot_titles=['Usefulness Ratings by Condition', 'Satisfying Ratings by Condition'])

# Define colors for each condition
colors = {
    'Functional UI': '#8BC1F7',
    'No UI': '#F9E0A2',
    'Placebo UI': '#C9190B'
}

# Create box plots for each score category
for i, category in enumerate(score_categories, start=1):
    for condition in combined_acceptance_df['condition'].unique():
        df_condition = combined_acceptance_df[combined_acceptance_df['condition'] == condition]
        fig.add_trace(
            go.Box(
                y=df_condition[category],
                name=condition,
                legendgroup=condition,
                showlegend=i == 1,  # Show legend only for the first subplot
                marker_color=colors[condition]
            ),
            row=1, col=i
        )

# Calculate overall y-axis range
y_min = combined_acceptance_df[score_categories].min().min()
y_max = combined_acceptance_df[score_categories].max().max()
y_range = [y_min - 0.5, y_max + 0.5]  # Add some padding

# Customize the layout
fig.update_layout(
    title='Acceptance Scale Ratings by Condition',
    plot_bgcolor='white',
    paper_bgcolor='white',
    boxmode='group',
    font=dict(size=12),
    height=500,
    width=1000,
    legend_title_text='Condition',
    # legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1)
    legend=dict(orientation="v", yanchor="top", y=1, xanchor="left", x=1.05),
    boxgroupgap=0.1
)

# Update x-axis and y-axis labels, and set fixed y-axis range
for i in range(1, 3):
    fig.update_xaxes(title_text="Condition", row=1, col=i)
    fig.update_yaxes(title_text=score_categories[i-1].capitalize(), row=1, col=i, range=y_range)

# Make boxes wider
fig.update_traces(width=0.6)

fig.show()

# Display ANOVA results
for category, results in anova_results.items():
    print(f"ANOVA results for {category}: F-statistic = {results['f_stat']:.4f}, P-value = {results['p_value']:.4f}")

# Interpret the results
alpha = 0.05
for category, results in anova_results.items():
    if results['p_value'] < alpha:
        print(f"{category}: Significant differences found (p-value = {results['p_value']:.4f})")
    else:
        print(f"{category}: No significant differences found (p-value = {results['p_value']:.4f})")

ANOVA results for usefulness: F-statistic = 9.4866, P-value = 0.0003
ANOVA results for satisfying: F-statistic = 7.1712, P-value = 0.0018
usefulness: Significant differences found (p-value = 0.0003)
satisfying: Significant differences found (p-value = 0.0018)


In [ ]:
combined_acceptance_df

,ParticipantID,usefulness,satisfying,condition
0,0,1.8,2.00,Functional UI
1,2,2.0,2.00,Functional UI
2,4,1.2,1.50,Functional UI
3,6,0.8,0.75,Functional UI
4,8,0.2,-0.25,Functional UI
5,10,0.4,0.00,Functional UI
6,12,1.6,1.25,Functional UI
7,14,2.0,1.50,Functional UI
8,16,0.6,2.00,Functional UI
9,18,1.2,1.50,Functional UI


In [ ]:
# Calculate and display descriptive statistics
def calculate_metrics(df, category):
    metrics = df.groupby('condition')[category].agg(['mean', 'median', 'std', 'min', 'max', 'count'])
    return metrics

# Calculate metrics for usefulness and satisfying
usefulness_metrics = calculate_metrics(combined_acceptance_df, 'usefulness')
satisfying_metrics = calculate_metrics(combined_acceptance_df, 'satisfying')

# Display metrics
print("Usefulness Metrics:")
print(usefulness_metrics)

print("\nSatisfying Metrics:")
print(satisfying_metrics)


Usefulness Metrics:
                   mean  median       std  min  max  count
condition                                                 
Functional UI  1.115385     1.2  0.582884  0.0  2.0     26
No UI         -0.085714    -0.4  1.036902 -1.4  1.8     14
Placebo UI     0.728571     1.2  0.994159 -1.4  2.0     14

Satisfying Metrics:
                   mean  median       std   min  max  count
condition                                                  
Functional UI  1.192308    1.25  0.704655 -0.50  2.0     26
No UI          0.035714   -0.50  1.208373 -1.75  2.0     14
Placebo UI     0.910714    1.25  0.978653 -0.75  2.0     14
